# Regressão — Otimização Bayesiana com Optuna

Otimizamos os hiperparâmetros do `RandomForestRegressor` e do `XGBRegressor` usando o
[Optuna](https://optuna.org/) (otimização bayesiana via TPE), avaliando cada combinação por
validação cruzada **no conjunto de treino**. O conjunto de teste só é usado no final, uma
única vez, para a avaliação do modelo campeão.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import optuna
from optuna.visualization.matplotlib import plot_optimization_history, plot_param_importances

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor

sns.set_theme(style='whitegrid')
optuna.logging.set_verbosity(optuna.logging.WARNING)
RANDOM_STATE = 42


## 2. Carregar dados e separar treino/teste

In [ ]:
df = pd.read_csv('../data/pib_municipios.csv')

ALVO = 'pib_per_capita'
FEATURES_NUM = ['populacao', 'participacao_agropecuaria', 'participacao_industria',
                'participacao_servicos', 'participacao_administracao_publica']
FEATURES_CAT = ['uf', 'regiao']

X = df[FEATURES_NUM + FEATURES_CAT]
y = df[ALVO]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE)

preprocessador = ColumnTransformer(transformers=[
    ('num', StandardScaler(), FEATURES_NUM),
    ('cat', OneHotEncoder(handle_unknown='ignore'), FEATURES_CAT),
])

kfold = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


## 3. Otimização — RandomForestRegressor

A cada *trial*, o Optuna propõe um conjunto de hiperparâmetros, treinamos com validação
cruzada no treino e retornamos o RMSE médio (a minimizar).

In [ ]:
def objetivo_rf(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 600),
        'max_depth': trial.suggest_int('max_depth', 3, 25),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
    }
    modelo = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1, **params)
    pipe = Pipeline([('preprocessador', preprocessador), ('modelo', modelo)])
    rmse_scores = -cross_val_score(pipe, X_train, y_train, cv=kfold,
                                    scoring='neg_root_mean_squared_error', n_jobs=-1)
    return rmse_scores.mean()

estudo_rf = optuna.create_study(direction='minimize', study_name='rf_pib_per_capita',
                                 sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
estudo_rf.optimize(objetivo_rf, n_trials=40, show_progress_bar=True)

print('Melhor RMSE (CV):', estudo_rf.best_value)
print('Melhores parâmetros:', estudo_rf.best_params)


## 4. Otimização — XGBRegressor

In [ ]:
def objetivo_xgb(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 800),
        'max_depth': trial.suggest_int('max_depth', 2, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
    }
    modelo = XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1, verbosity=0, **params)
    pipe = Pipeline([('preprocessador', preprocessador), ('modelo', modelo)])
    rmse_scores = -cross_val_score(pipe, X_train, y_train, cv=kfold,
                                    scoring='neg_root_mean_squared_error', n_jobs=-1)
    return rmse_scores.mean()

estudo_xgb = optuna.create_study(direction='minimize', study_name='xgb_pib_per_capita',
                                  sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
estudo_xgb.optimize(objetivo_xgb, n_trials=40, show_progress_bar=True)

print('Melhor RMSE (CV):', estudo_xgb.best_value)
print('Melhores parâmetros:', estudo_xgb.best_params)


## 5. Histórico de otimização e importância dos hiperparâmetros

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_optimization_history(estudo_rf, ax=axes[0])
axes[0].set_title('RandomForest — histórico (Optuna)')
plot_optimization_history(estudo_xgb, ax=axes[1])
axes[1].set_title('XGBoost — histórico (Optuna)')
plt.tight_layout()
plt.savefig('../reports/09_optuna_historico_otimizacao.png', dpi=120)
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_param_importances(estudo_rf, ax=axes[0])
axes[0].set_title('RandomForest — importância dos hiperparâmetros')
plot_param_importances(estudo_xgb, ax=axes[1])
axes[1].set_title('XGBoost — importância dos hiperparâmetros')
plt.tight_layout()
plt.savefig('../reports/10_optuna_importancia_hiperparametros.png', dpi=120)
plt.show()


## 6. Escolher o campeão e avaliar no teste (uma única vez)

In [ ]:
candidatos = {
    'RandomForest_otimizado': (RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1, **estudo_rf.best_params),
                                estudo_rf.best_value),
    'XGBoost_otimizado': (XGBRegressor(random_state=RANDOM_STATE, n_jobs=-1, verbosity=0, **estudo_xgb.best_params),
                           estudo_xgb.best_value),
}

for nome, (_, rmse_cv) in candidatos.items():
    print(f'{nome:22s} | RMSE médio (CV) = {rmse_cv:,.2f}')

melhor_nome = min(candidatos, key=lambda n: candidatos[n][1])
melhor_modelo, _ = candidatos[melhor_nome]
print('\nModelo campeão:', melhor_nome)


In [ ]:
pipeline_final = Pipeline([
    ('preprocessador', preprocessador),
    ('modelo', melhor_modelo),
])
pipeline_final.fit(X_train, y_train)

pred_teste = pipeline_final.predict(X_test)

rmse = mean_squared_error(y_test, pred_teste, squared=False)
mae = mean_absolute_error(y_test, pred_teste)
r2 = r2_score(y_test, pred_teste)

print(f'Avaliação final no conjunto de teste — {melhor_nome}')
print(f'RMSE = {rmse:,.2f}')
print(f'MAE  = {mae:,.2f}')
print(f'R2   = {r2:.3f}')


## 7. Previsto vs. real e resíduos (modelo final)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, pred_teste, alpha=0.4)
lim = [y_test.min(), y_test.max()]
axes[0].plot(lim, lim, color='red', linestyle='--')
axes[0].set_xlabel('Real')
axes[0].set_ylabel('Previsto')
axes[0].set_title(f'Previsto vs. Real — {melhor_nome}')

residuos = y_test - pred_teste
sns.histplot(residuos, kde=True, ax=axes[1])
axes[1].set_title('Distribuição dos resíduos — modelo final')

plt.tight_layout()
plt.savefig('../reports/11_modelo_final_previsto_vs_real.png', dpi=120)
plt.show()


## 8. Salvar o modelo final

In [ ]:
joblib.dump(pipeline_final, '../models/modelo_final.joblib')
print(f'Modelo final ({melhor_nome}) salvo em ../models/modelo_final.joblib')


## 9. Resumo do projeto

| Etapa | Notebook | Resultado |
|---|---|---|
| EDA | `01_EDA.ipynb` | Entendimento do alvo e das features |
| Baseline | `02_Baseline.ipynb` | Primeiro RMSE/MAE/R2 de referência |
| Comparação + CV | `03_Comparacao_Modelos_CV.ipynb` | Ranking de modelos por validação cruzada |
| Otimização | `04_Otimizacao_Optuna.ipynb` | Hiperparâmetros ajustados via Optuna e avaliação final no teste |

O modelo final está salvo em `../models/modelo_final.joblib` e pode ser recarregado com
`joblib.load(...)` para gerar novas previsões.